In [1]:
import sys
import os
import copy
import shutil

import numpy as np
import matplotlib.pyplot as plt

# Add the src directory to the path. TEMPORARY FIX
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.predictor import ShorelinePredictor

from src.data_processing.dataset_loader import CoastData

from src.data_postprocessing import obtain_shoreline

from typing import Type

import json

c:\Users\josep\.conda\envs\imagine\Lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
# Execute this cell to make sure 
# that external modules are reloaded
%load_ext autoreload
%autoreload 2

In [3]:
image_type_paths = {
    # "oblique": {
    #     "path": os.path.abspath(os.path.join(os.getcwd(), "../../data/SCLabels_oblique_registered_v1.0.0/")),
    #     "num_classes": 2,
    #     "weights_path": os.path.abspath(os.path.join(os.getcwd(), "../../artifacts/article/experiment1/oblique"))
    # },
    "rectified": {
        "path": os.path.abspath(os.path.join(os.getcwd(), "../../data/SCLabels_v1.0.0/")),
        "num_classes": 1,
        "weights_path": os.path.abspath(os.path.join(os.getcwd(), "../../artifacts/article/experiment1/rectified"))
    }
}

networks: dict[str] = {
    "BiLSTM": {
        "weights_path": {
            "rectified": "2025-10-27-13-47-09_rectified_BiLSTM",
            "oblique": "todo"
        },
        "patch": {
            "patch_size": (256, 256),
            "stride": (128, 128)
        }
    }
}

output_dir = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment1/"))

In [5]:
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

for data_type in image_type_paths:
    print(f"\n{'#'*30}\nProcessing {data_type} images\n{'#'*30}")
    # Create folder
    output_subdir = os.path.join(output_dir, data_type)
    if not os.path.exists(output_subdir):
        os.makedirs(output_subdir)
        print(f"Created output directory: {output_subdir}")
        
    data_path = image_type_paths[data_type]["path"]
    num_classes = image_type_paths[data_type]["num_classes"]
    weights_path = image_type_paths[data_type]["weights_path"]

    print(f"Data path: {data_path}")

    # Load data
    data = CoastData(data_path)

    get_coords = True if data_type == "oblique" else False
    get_mask = True if data_type == "rectified" else False
    filtered_data = data.split_data(get_metadata=True, get_coords=get_coords, get_mask=get_mask)

    print(f"Number of samples: {len(filtered_data['test']['images'])}")
    for network in networks:
        # Create images/masks/predictions folders
        base_output_dir = os.path.join(output_subdir, network)
        images_output_dir = os.path.join(base_output_dir, "images")
        masks_output_dir = os.path.join(base_output_dir, "masks")
        predictions_output_dir = os.path.join(base_output_dir, "predictions")
        predicted_mask_output_dir = os.path.join(base_output_dir, "predicted_masks")

        for dir_path in [images_output_dir, masks_output_dir, predictions_output_dir, predicted_mask_output_dir]:
            if not os.path.exists(dir_path):
                os.makedirs(dir_path)
                print(f"Created directory: {dir_path}")

        print(f"\n{'-'*20}\nPredicting with {network} - {data_type}\n{'-'*20}")
        net_weights_path = os.path.join(weights_path, networks[network]["weights_path"][data_type], "models/best_model.pth")

        all_metadata_list = []
        
        predictor = ShorelinePredictor(network, net_weights_path, num_classes)

        counter = 0
        total_images = len(filtered_data['test']['images'])

        # Predict only the test set
        for path_img, path_mask, metadata in zip(filtered_data['test']['images'], filtered_data['test']['masks'], filtered_data['test']['metadata']):
            # Get filenames
            img_filename = os.path.basename(path_img)
            mask_filename = os.path.basename(path_mask) if data_type == "rectified" else img_filename.replace("image", "mask")
            output_img_path = os.path.join(images_output_dir, img_filename)
            output_mask_path = os.path.join(masks_output_dir, mask_filename)

            # Predict
            if data_type == "oblique":
                shoreline_coords = metadata['image']['shoreline']['coordinates']
                output = predictor.predict_oblique_with_coords(path_img, shoreline_coords=shoreline_coords, patch_size=networks[network]["patch"]["patch_size"], stride=networks[network]["patch"]["stride"], extract_mask_coords=True)

            elif data_type == "rectified":
                output = predictor.predict_rectified_with_mask(path_img, path_mask, patch_size=networks[network]["patch"]["patch_size"], stride=networks[network]["patch"]["stride"], extract_mask_coords=True, landward_pixel=0, seaward_pixel=1)

            # Copy original image and mask to output folders
            if not os.path.exists(output_img_path):
                shutil.copy(path_img, output_img_path)
            if not os.path.exists(output_mask_path) and data_type == "rectified":
                shutil.copy(path_mask, output_mask_path)

            # Save prediction and predicted mask
            prediction_filename = mask_filename.replace("mask", f"prediction")
            output_prediction_path =  os.path.join(predictions_output_dir, prediction_filename)
            plt.imsave(output_prediction_path, output['predicted_image'])

            predicted_mask_filename = mask_filename.replace("mask", f"predicted_mask")
            output_predicted_mask_path =  os.path.join(predicted_mask_output_dir, predicted_mask_filename)
            plt.imsave(output_predicted_mask_path, output['predicted_mask'], cmap='gray')

            # Update metadata
            meta_copy = copy.deepcopy(metadata)
            meta_copy.setdefault("image", {})
            meta_copy["image"].setdefault("shoreline", {})

            patch = networks.get(network, {}).get('patch', {})
            patch_size = patch.get('patch_size', 'unknown')
            stride = patch.get('stride', 'unknown')

            meta_copy['image']['predicted_mask'] = {
                "filename": predicted_mask_filename,
                "description": f"Patch size {patch_size} and stride {stride}",
                "labels": {
                    "0": "NoData",
                    "75": "Landwards",
                    "150": "Seawards",
                    "255": "Shoreline"
                }
            }

            meta_copy['image']['prediction'] = {
                "filename": prediction_filename,
                "description": f"Predicted shoreline overlaid on original image by model (e.g., {network}), patched prediction with patch size {patch_size} and stride {stride}. Original shoreline shown in green, predicted shoreline shown in red and overlapping areas in yellow."
            }

            meta_copy["image"]["shoreline"]["coordinates"] = {
                "u": output.get('original_shoreline_coords', {}).get('u', []),
                "v": output.get('original_shoreline_coords', {}).get('v', [])
            }
            meta_copy["image"]["shoreline"]["description"] = "uv coordinates of the shoreline..."

            meta_copy['image']['predicted_shoreline'] = {
                "description": f"PREDICTED shoreline by model (e.g., {network}), patched prediction with patch size {patch_size} and stride {stride}.",
                "coordinates": {
                    "u": output.get('shoreline_coords', {}).get('u', []),
                    "v": output.get('shoreline_coords', {}).get('v', [])
                }
            }

            all_metadata_list.append(meta_copy)

            counter += 1
            if counter % max(1, total_images // 10) == 0 or counter == total_images:
                print(f"Processed {counter}/{total_images} images ({(counter/total_images)*100:.1f}%)")

        # Save metadata to JSON
        metadata_output_path = os.path.join(base_output_dir, "metadata.json")
        with open(metadata_output_path, 'w') as json_file:
            json.dump(all_metadata_list, json_file, indent=4)


##############################
Processing rectified images
##############################
Data path: c:\Users\josep\Documents\SOCIB\Shoreline-extraction\data\SCLabels_v1.0.0
CoastData: global - 1717 images
Coast: agrelo, Total size: 244
Coast: arenaldentem, Total size: 40
Coast: cadiz, Total size: 946
Coast: cies, Total size: 430
Coast: samarador, Total size: 57
Number of samples: 174

--------------------
Predicting with BiLSTM - rectified
--------------------
Processed 17/174 images (9.8%)
Processed 34/174 images (19.5%)
Processed 51/174 images (29.3%)
Processed 68/174 images (39.1%)
Processed 85/174 images (48.9%)
Processed 102/174 images (58.6%)
Processed 119/174 images (68.4%)
Processed 136/174 images (78.2%)
Processed 153/174 images (87.9%)
Processed 170/174 images (97.7%)
Processed 174/174 images (100.0%)
